[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skous2/is455-w2027/blob/main/module_04/Worksheet_4A_Getting_and_Cleaning_Data.ipynb)

# Worksheet 4A · Getting and cleaning data

**Optional and not graded.** Work through it before Tuesday's class (Feb 2). Tuesday's walk-through covers sections 1–5 on getting data from a JSON file, a database and an API, and Thursday's class covers sections 6–10 on cleaning it. Expect about 40–50 minutes.

**Before you start,** choose **File → Save a copy in Drive** so your work is saved. Try each problem yourself first. When you get stuck, ask your AI assistant for hints instead of the answer, the way the tutor prompt in Modules 1 and 2 did. Write answers in your own words.

Each section starts with a short worked example from a made-up smoothie shop, then gives you the same kind of problem for Canyon Fitness. Each section also links the reading that covers it, and you can read each one when you reach its section.

## Where the data comes from

The clean Canyon Fitness file you've used so far had to come from somewhere. Company data usually lives in several places, such as JSON files, databases and the member system's **API**, a web address that returns data when your code requests it. This week you pull Canyon Fitness data from all three and then clean it.

For this class, every source is saved as a file on the course GitHub, so everyone gets the same data and nothing can go down during class. Your code works exactly as it would with a live system, because you still request a web address and get data back.

## 1. What JSON looks like

*McKinney 6.1, "Reading and Writing Data in Text Format," covers this under "JSON Data."*

**Worked example.** **JSON** is text built from two pieces. An **object**, written with `{ }`, holds named values and becomes a Python dictionary. An **array**, written with `[ ]`, holds a list of values and becomes a Python list. Objects and arrays can sit inside each other. JSON writes `true`, `false` and `null`, which Python reads as `True`, `False` and `None`. `json.loads` turns JSON text into Python, and `json.dumps` turns Python back into JSON text.

In [ ]:
import json

text = """
[
  {"drink": "Mango", "price": 4.75, "seasonal": false, "toppings": ["chia", "coconut"], "supplier": {"name": "Valley Farms", "city": "Orem"}},
  {"drink": "Berry", "price": 5.25, "seasonal": true,  "toppings": [],                  "supplier": null}
]
"""
menu = json.loads(text)
print(type(menu), type(menu[0]))                   # the array became a list, each object a dictionary
print(menu[0]["supplier"]["city"])                 # step into a nested object
print(menu[1]["seasonal"], menu[1]["supplier"])    # true became True, null became None
print([d["drink"] for d in menu if "chia" in d["toppings"]])
print(json.dumps(menu[0], indent=2))               # JSON text again, indented so it's easy to read

**Your turn.**

Run the cell below to download the gym file as text. Section 3 explains how `requests.get` works.

In [ ]:
import json
import requests
import pandas as pd

JSON_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/json/canyon_gyms.json"
gyms_text = requests.get(JSON_URL, timeout=10).text
print(gyms_text[:500])

**1a.** Turn `gyms_text` into Python with `json.loads` and store it as `gyms`. Print the type of `gyms`, the type of its first item, and how many gyms there are. Then loop through `gyms` and print each gym's name, its city from the nested `address`, and how many `amenities` it lists.

In [ ]:
# Your code here


**1b.** Use list comprehensions to list the gyms whose `amenities` include `"pool"`, and the gyms whose `manager` is `null` in the file. What does `null` become in Python?

In [ ]:
# Your code here


*Your answer here.*

**1c.** Why would `amenities` be awkward to store in a CSV file?

*Your answer here.*

## 2. Query a SQLite database from pandas

*McKinney 6.4, "Interacting with Databases," covers `sqlite3` and `pd.read_sql`.*

**Worked example.** **SQLite** is a database stored in a single file. You save the file, open a **connection** with `sqlite3.connect`, and pass SQL to `pd.read_sql_query`, which returns the result as a DataFrame you can keep working with. Every SQLite database lists its tables in `sqlite_master`. When a query depends on a value, write a `?` placeholder and pass the value in `params` instead of building the SQL with an f-string, so an unexpected value can't change the query. SQLite has no True/False type, so it stores them as 1 and 0.

In [ ]:
import sqlite3
import requests

db_url = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/sqlite/smoothie.db"
with open("smoothie.db", "wb") as f:                # save the database file into Colab
    f.write(requests.get(db_url, timeout=10).content)

conn = sqlite3.connect("smoothie.db")
display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", conn))

query = """
SELECT o.drink, COUNT(*) AS orders, SUM(c.member) AS member_orders
FROM orders o
JOIN customers c ON o.customer = c.customer
GROUP BY o.drink
"""
by_drink = pd.read_sql_query(query, conn)            # the result arrives as a DataFrame
by_drink["member_share"] = by_drink["member_orders"] / by_drink["orders"]
display(by_drink)

display(pd.read_sql_query("SELECT * FROM orders WHERE drink = ?", conn, params=("Mango",)))
conn.close()

**Your turn.**

Run the cell below to save the Canyon Fitness database into Colab and connect to it.

In [ ]:
import sqlite3

DB_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/sqlite/canyon_fitness.db"
with open("canyon_fitness.db", "wb") as f:
    f.write(requests.get(DB_URL, timeout=10).content)

conn = sqlite3.connect("canyon_fitness.db")

**2a.** List the database's tables, then show the first five rows of each one.

In [ ]:
# Your code here


**2b.** Write one SQL query that returns each gym with its number of members and its number of cancellations, using a `LEFT JOIN` from `members` to `cancellations`. Read the result into a DataFrame named `by_gym`. Then, in pandas, add a `cancel_rate` column and sort from highest to lowest.

In [ ]:
# Your code here


**2c.** Why does this query need a `LEFT JOIN` instead of a plain `JOIN`? And when would you do work in SQL before the data reaches pandas?

*Your answer here.*

## 3. Make one request

*Read [requests Quickstart, "Make a Request," "JSON Response Content" and "Response Status Codes"](https://requests.readthedocs.io/en/latest/user/quickstart/). McKinney 6.3, "Interacting with Web APIs", uses the same tools.*

**Worked example.** `requests.get` asks a web address for data. The response's **status code** says whether it worked. 200 means success and 404 means the address doesn't exist. `.json()` turns the JSON text into Python dictionaries and lists. `timeout=10` stops your code from waiting forever if the server doesn't answer.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/smoothie/orders_page_1.json"
response = requests.get(url, timeout=10)
print(response.status_code)      # 200 means the request worked

page = response.json()           # JSON text becomes a Python dictionary
print(page.keys())
print(page["results"][0])        # the first order on this page

**Your turn.**

Run the cell below to store the address of the Canyon Fitness API.

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/canyon/"
first_url = BASE_URL + "members_page_1.json"

**3a.** Request `first_url`. Print the status code, the keys of the response, `total_records`, `total_pages`, and the first member in `results`.

In [ ]:
# Your code here


**3b.** Request `BASE_URL + "members_page_99.json"`, a page that doesn't exist, and print its status code. Don't call `.json()` on it. What does the code mean, and why should you check the status code before calling `.json()`?

In [ ]:
# Your code here


*Your answer here.*

**API keys** are password-like strings that most real APIs require, to identify you and count your requests. Never type a key into a notebook cell, because notebooks get shared and pushed to GitHub, where automated tools scan for leaked keys. In Colab, save the key in the **Secrets** panel, the key icon in the left sidebar, and read it in code with `from google.colab import userdata` and then `api_key = userdata.get("MY_API_KEY")`. If a key does leak, revoke it with the provider and get a new one.

## 4. Get every page

*Read [requests Quickstart, "Passing Parameters In URLs" and "Timeouts"](https://requests.readthedocs.io/en/latest/user/quickstart/).*

**Worked example.** APIs usually return large results in pages. This one puts the address of the next page in each response's `"next"` field, and the last page has `"next": null`, which Python reads as `None`. A `while` loop keeps requesting pages until there isn't a next one. `raise_for_status()` stops with an error if a request fails, and `time.sleep` pauses between requests so you stay under the API's **rate limit**, the number of requests it allows in a given time. Many other APIs take the page number as a parameter instead, such as `requests.get(url, params={"page": 2})`.

In [ ]:
import time

orders = []
url = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/api/smoothie/orders_page_1.json"
while url is not None:
    response = requests.get(url, timeout=10)
    response.raise_for_status()          # stop here if the request failed
    page = response.json()
    orders.extend(page["results"])       # add this page's records to the list
    url = page["next"]                   # None on the last page, which ends the loop
    time.sleep(0.2)                      # pause so you don't overload the server

print(len(orders))

**Your turn.**

**4a.** Collect every member record from every page into a list called `records`. Print how many records you collected and compare it with `total_records` from page 1.

In [ ]:
# Your code here


**4b.** Why pause between requests? What could happen to a script that sends thousands of requests as fast as it can?

*Your answer here.*

## 5. Turn JSON into a DataFrame

*Read the [pandas documentation for `json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).*

**Worked example.** Each record is a dictionary, and some values are dictionaries themselves. `pd.json_normalize` turns a list of records into a DataFrame and flattens nested fields into columns with dots in their names, such as `customer.name`. `rename` then gives the columns simpler names.

In [ ]:
orders_df = pd.json_normalize(orders)
display(orders_df)

orders_df = orders_df.rename(columns={"customer.name": "customer", "customer.member": "member"})
print(list(orders_df.columns))

**Your turn.**

**5a.** Turn `records` into a DataFrame named `members` with `pd.json_normalize`. Rename these columns so they match the clean file you used in Modules 2 and 3, then print the shape and the column names.

| API name | New name |
| --- | --- |
| `joined` | `join_date` |
| `plan.name` | `plan` |
| `plan.monthly_fee` | `monthly_fee` |
| `visits.last_30_days` | `visits_last_30_days` |
| `visits.prior_30_days` | `visits_prior_30_days` |
| `visits.days_since_last` | `days_since_last_visit` |

In [ ]:
# Your code here


## 6. Duplicate records

*McKinney 7.2, "Data Transformation," covers this under "Removing Duplicates."*

**Worked example.** `duplicated()` marks each row that repeats an earlier row exactly, so `.sum()` counts them. Checking the ID column alone catches records that repeat an ID but differ somewhere else. `drop_duplicates()` keeps the first copy of each exact duplicate.

In [ ]:
sales = pd.DataFrame({"order_id": [1, 2, 2, 3], "drink": ["Mango", "Berry", "Berry", "Mango"]})
print(sales.duplicated().sum())               # rows that exactly repeat an earlier row
print(sales["order_id"].duplicated().sum())   # IDs that appear more than once
sales = sales.drop_duplicates()
print(len(sales))

**Your turn.**

**6a.** Count the exact duplicate rows in `members` and the repeated `member_id` values. Drop the exact duplicates, then confirm that every `member_id` is now unique and print the number of rows.

In [ ]:
# Your code here


## 7. Text that should be numbers or dates

*McKinney 7.4, "String Manipulation," covers `.str` functions, and McKinney 11.1 covers converting text to dates.*

**Worked example.** A dollar sign makes pandas read a price as text, so math on it fails. `.str.replace` removes the sign and `.astype(float)` turns the rest into a number. Dates written two different ways need `pd.to_datetime` with `format="mixed"`, which reads each value's format separately.

In [ ]:
prices = pd.DataFrame({"price": ["$4.75", "$5.25"], "date": ["2026-09-28", "09/30/2026"]})
prices["price"] = prices["price"].str.replace("$", "").astype(float)
prices["date"] = pd.to_datetime(prices["date"], format="mixed")
print(prices.dtypes)
prices

**Your turn.**

**7a.** Count how many `join_date` values contain a `/`, which marks the month/day/year format. Then convert `monthly_fee` to a number and convert `join_date`, `next_visit_date` and `cancel_date` to dates. Print the data types to check.

In [ ]:
# Your code here


## 8. Inconsistent categories and yes/no text

*McKinney 7.4 covers `.str.strip`, `.str.title` and `.str.upper`. McKinney 7.2 covers changing values under "Transforming Data with a Function or Mapping."*

**Worked example.** `value_counts` exposes spellings that should be the same category. `.str.strip()` removes spaces at the ends and `.str.title()` capitalizes the first letter of each word. Yes/no text becomes `True`/`False` by checking whether the uppercase value starts with "Y".

In [ ]:
drinks = pd.DataFrame({"drink": ["Mango", "mango ", "MANGO", "Berry"], "member": ["Y", "N", "Yes", "no"]})
print(drinks["drink"].value_counts())

drinks["drink"] = drinks["drink"].str.strip().str.title()
drinks["member"] = drinks["member"].str.upper().str.startswith("Y")
print(drinks["drink"].value_counts())
drinks

**Your turn.**

**8a.** Show `value_counts` for `home_gym`, then fix it so each gym has one spelling and show `value_counts` again. Then convert `autopay` to `True`/`False`.

In [ ]:
# Your code here


## 9. Placeholder codes and impossible values

*McKinney 7.2 covers this under "Replace Values" and "Detecting and Filtering Outliers." McKinney 7.1 covers missing data.*

**Worked example.** Some systems store a **placeholder code** such as -1 or 999 when a value is unknown. Left in place, a placeholder distorts every average, so it should become missing (`NaN`). An **impossible value**, such as an age of 240, can't be fixed by guessing, so it becomes missing too. A combination can also be impossible even when each value looks fine, such as a refund dated before its order. `.loc[condition, column] = ...` changes only the rows where the condition is `True`.

In [ ]:
import numpy as np

sales = pd.DataFrame({
    "age":         [34, -1, 27, 240],
    "order_date":  pd.to_datetime(["2026-09-28", "2026-09-28", "2026-09-29", "2026-09-30"]),
    "refund_date": pd.to_datetime([None, "2026-09-20", None, None]),
})
sales.loc[(sales["age"] < 0) | (sales["age"] > 110), "age"] = np.nan        # -1 means unknown; 240 is a typo
sales.loc[sales["refund_date"] < sales["order_date"], "refund_date"] = pd.NaT   # a refund can't come before the order
sales

**Your turn.**

**9a.** Count the members whose `age` is below 0 or above 110, then set those ages to missing.

In [ ]:
# Your code here


**9b.** A member can't visit after cancelling. Count the members whose `next_visit_date` is after their `cancel_date`, then set those `next_visit_date` values to missing. Why is it safer to blank the visit date than to delete the member?

In [ ]:
# Your code here


*Your answer here.*

## 10. Check your work

Run the cell below. It loads the clean file from Modules 2 and 3 and compares it with your cleaned data, column by column. If your cleaning matches, every column shows 0 differences except `age`, where the ages you set to missing differ on purpose. If another column shows differences, go back to the section that cleans it.

In [ ]:
clean = pd.read_csv("https://raw.githubusercontent.com/skous2/is455-w2027/main/data/canyon_fitness_members.csv", parse_dates=["join_date", "next_visit_date", "cancel_date"])
print(f"Your rows: {len(members)}   Clean file rows: {len(clean)}")

if len(members) == len(clean):
    mine = members.sort_values("member_id").reset_index(drop=True)
    for col in clean.columns:
        same = (mine[col] == clean[col]) | (mine[col].isna() & clean[col].isna())
        print(f"{col:30} {(~same).sum():>4} rows differ")

## Bring to class

Write down anything that confused you. Tuesday's walk-through covers sections 1–5, Thursday's class covers sections 6–10, and the answer key is posted after Tuesday's class.

*Your answer here.*